In [1]:
import sys, shutil
from pathlib import Path
import cv2, numpy as np, pandas as pd, torch
from ultralytics import YOLO

PROJECT = Path.cwd().parent
sys.path.append(str(PROJECT / "src"))
from football_eval import iou, load_gt

DS = PROJECT / "data" / "detection"
REVIEW = PROJECT / "outputs" / "02b_goalkeepers" / "review"
REVIEW.mkdir(parents=True, exist_ok=True)
DEVICE = 0 if torch.cuda.is_available() else "cpu"
model = YOLO("yolo11m.pt")

In [2]:
BACKUP = PROJECT / "data" / "detection_labels_backup"
if not BACKUP.exists():
    for split in ["train", "valid", "test"]:
        shutil.copytree(DS / split / "labels", BACKUP / split / "labels")
    print("labels backed up")
else:
    print("backup already exists")

labels backed up


In [5]:
rows, n = [], 0
for split in ["train", "valid", "test"]:
    for img_path in sorted((DS / split / "images").glob("*")):
        lbl = DS / split / "labels" / (img_path.stem + ".txt")
        if not lbl.exists():
            continue
        img = cv2.imread(str(img_path)); H, W = img.shape[:2]
        res = model.predict(img, imgsz=1280, conf=0.5, classes=[0], device=DEVICE, verbose=False)[0]
        gts = load_gt(lbl, W, H, {1, 2})
        for x1, y1, x2, y2 in res.boxes.xyxy.cpu().numpy():
            if (y2 - y1) < 70 or y2 < 0.25 * H:        # skip tiny boxes and the stands area
                continue
            if any(iou(g, (x1, y1, x2, y2)) >= 0.3 for g in gts):   # already labeled
                continue
            n += 1
            cid = f"cand_{n:04d}"
            pad = 40
            crop = img[max(0, int(y1) - pad):int(y2) + pad, max(0, int(x1) - pad):int(x2) + pad]
            cv2.imwrite(str(REVIEW / f"{cid}.jpg"), crop)
            rows.append(dict(cid=cid, split=split, stem=img_path.stem,
                             x1=x1, y1=y1, x2=x2, y2=y2, W=W, H=H))

cands = pd.DataFrame(rows)
cands.to_csv(REVIEW.parent / "candidates.csv", index=False)
print(len(cands), "candidate crops saved in", REVIEW)

59 candidate crops saved in c:\Users\seree\OneDrive\Desktop\FalseNine\outputs\02b_goalkeepers\review


In [6]:
import sys
from pathlib import Path
import pandas as pd

PROJECT = Path.cwd().parent
sys.path.append(str(PROJECT / "src"))
from football_eval import iou, load_gt

DS = PROJECT / "data" / "detection"
REVIEW = PROJECT / "outputs" / "02b_goalkeepers" / "review"
cands = pd.read_csv(REVIEW.parent / "candidates.csv")

kept = {p.stem for p in REVIEW.glob("*.jpg")}
print(len(kept), "crops kept out of", len(cands), "candidates")

added = 0
for _, r in cands[cands["cid"].isin(kept)].iterrows():
    lbl = DS / r["split"] / "labels" / (r["stem"] + ".txt")
    box = (r["x1"], r["y1"], r["x2"], r["y2"])
    if any(iou(g, box) >= 0.5 for g in load_gt(lbl, r["W"], r["H"], {0, 1, 2})):
        continue                                   # already labeled, so rerunning is safe
    cx = (r["x1"] + r["x2"]) / 2 / r["W"]; cy = (r["y1"] + r["y2"]) / 2 / r["H"]
    w = (r["x2"] - r["x1"]) / r["W"];      h = (r["y2"] - r["y1"]) / r["H"]
    txt = lbl.read_text()
    if txt and not txt.endswith("\n"):
        txt += "\n"
    lbl.write_text(txt + f"1 {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}\n")   # class 1 = Player
    added += 1
print("added", added, "boxes")

80 crops kept out of 59 candidates
added 58 boxes
